# Task Retries, Stage Retries, and Failure Handling

**Module**: 11_fault_tolerance_and_reliability  
**Notebook**: 03_task_retries.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Task retry mechanisms, spark.task.maxFailures, stage retries, and handling transient exceptions.
- **Why does this topic exist?**: Transient errors (network hiccups, disk blips) should not abort long-running ETL jobs.
- **What problem does it solve?**: Understanding task retry limits and avoiding poison pill data crashes.
- **Where is it used in real-world Data Engineering?**: Production resilience engineering.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Task scheduling, executor failure, Spark UI |
| **Required depth** | Working knowledge: can explain task and stage retry behavior |
| **Data Engineer** | Configure retries and distinguish transient from deterministic failures. |
| **Senior Data Engineer** | Diagnose retry loops and protect against poison records. |
| **Spark specialist** | Reason about task attempt limits, stage attempts, and fetch-failure handling. |
| **Next topic** | `11_fault_tolerance_and_reliability/04_checkpointing.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Spark retries failed tasks and stages to survive transient failures. Retries help with network blips,
speculative stragglers, and lost executors; they do not help with deterministic bugs such as a malformed record that
crashes every attempt. A retry loop is therefore a signal to inspect the failure, not just to increase the retry
count.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Task retry** | Rerun a failed task | Controlled by `spark.task.maxFailures` |
| **Stage retry** | Rerun a failed stage | Controlled by `spark.stage.maxConsecutiveAttempts` |
| **Fetch failure** | Reducer cannot read a map output | Handled specially; may rerun the map stage |
| **Speculation** | Duplicate slow tasks | `spark.speculation`, disabled by default |
| **Poison record** | Deterministic failure | Fails every retry |
| **Blacklisting** | Avoid bad executors/nodes | Can reduce repeated failures |

### How it works, step by step

1. A task attempt fails and reports the failure to the scheduler.
2. The scheduler checks the attempt count and the failure type.
3. If attempts remain, it resubmits the task, preferably on another executor.
4. If the stage loses shuffle outputs, it may resubmit parent tasks.
5. After the maximum attempts, the stage or job fails.
6. Persistent failures usually indicate a deterministic bug or a bad node.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Task_Retries_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
print("spark.task.maxFailures: ", spark.conf.get("spark.task.maxFailures", "4 (default)"))

---

## Experiment: Simulating a Task Retry with an Unstable UDF

- **1. What are we testing?**: Run a function that fails on attempt 0 and succeeds on attempt 1.
- **2. What do we expect?**: Spark retries the task automatically up to maxFailures.

In [ ]:
from pyspark import TaskContext

def flaky_task(x):
    ctx = TaskContext.get()
    attempt = ctx.attemptNumber()
    if x == 42 and attempt == 0:
        raise RuntimeError("Transient blip on attempt 0!")
    return f"Processed {x} on attempt {attempt}"

try:
    res = spark.sparkContext.parallelize([1, 42, 3], 2).map(flaky_task).collect()
    print("Flaky task succeeded after retry:")
    for r in res:
        print(" ", r)
except Exception as e:
    print(f"Task retry failed: {e}")

**3. What happened?**: Task failed on attempt 0 and automatically succeeded on attempt 1.  
**4. Why did it happen?**: TaskScheduler re-queued the task because attemptNumber < spark.task.maxFailures.  
**5. Practical Engineering Lesson**: Spark task retries absorb transient network and I/O glitches seamlessly.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Flaky network shuffle

- **Industry**: Telecom
- **Scenario**: A shuffle stage occasionally fails with fetch errors during peak network congestion.
- **Problem**: Transient fetch failures can be retried, but repeated failures may indicate a lost executor.
- **Why this concept is relevant**: Retry settings plus an external shuffle service make the job resilient to transient loss.
- **Implementation idea**: Inspect fetch-failure counts, stage attempts, and executor loss; tune timeouts rather than retries alone.
- **Expected result**: A job that survives transient congestion without masking a real problem.
- **Alternative**: Increase task retries.
- **Trade-offs**: More retries help only if the failure is transient; they can prolong a deterministic failure.

### Use Case 2: Poison record in a parser

- **Industry**: Retail
- **Scenario**: A malformed record crashes the task on every attempt.
- **Problem**: The job retries until the maximum, then fails, wasting time.
- **Why this concept is relevant**: The failure is deterministic; the fix is to validate/quarantine the record, not to retry more.
- **Implementation idea**: Inspect the task error, reproduce on the record, and add validation or a dead-letter path.
- **Expected result**: A job that handles bad records explicitly.
- **Alternative**: Skip the record silently.
- **Trade-offs**: Skipping hides data loss; quarantine preserves the record for investigation.

---

## Business Value

- Higher resilience to transient failures.
- Faster diagnosis of deterministic failures.
- Less wasted compute on retry loops.
- Clear operational thresholds for alerts.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Distinguish transient from deterministic failures before tuning retries.
- Monitor task and stage attempt counts.
- Use blacklisting or node maintenance for repeated bad nodes.
- Do not set retries so high that failures are hidden.
- Test failure handling with a deliberate bad record.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Force a task failure

**Steps**

1. Create a UDF that throws on a specific key.
2. Run the job and observe task attempts and the final failure.
3. Fix the UDF or quarantine the key, then rerun.

**What to observe**: A deterministic failure exhausts retries and fails the stage; the UI shows repeated attempts.

**Healthy result**: The failure is diagnosed and handled explicitly.

**Problematic result**: The job retries the same record many times.

**Engineering takeaway**: Retries are for transient faults, not data validation.

---

## Trade-offs and Alternatives

### Advantages

- Automatic resilience
- Configurable limits
- Handles executor loss

### Limitations

- Cannot fix deterministic bugs
- Retries add latency
- Can mask systemic issues

### When to use it

- Transient network failures
- Spot-instance churn
- Large shuffles

### When not to use it

- Infinite retries
- Ignoring repeated failures
- Using retries instead of validation

### Alternatives inside Spark

- Task retries
- Stage retries
- Speculation
- Blacklisting

### Alternatives outside Spark

- Workflow retries
- Circuit breakers
- Dead-letter queues

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Max Failures
**Problem Statement**: Print spark.task.maxFailures.  
**Input / Context**: conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Poison Pill Concept
**Problem Statement**: Explain what a poison pill record is.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Stage Abort Rule
**Problem Statement**: When is an entire stage aborted?  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The retry storm

**Scenario**: A stage retries for hours and eventually fails, consuming the whole cluster.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the task error and whether it is deterministic.
2. Look for a poison record or a missing dependency.
3. Reduce retries or fail fast while debugging.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Max Failures):
```python
print(spark.conf.get('spark.task.maxFailures','4'))
```

#### Solution 2 (Poison Pill Concept):
```python
print('A bad record that deterministically crashes on every retry until maxFailures is exceeded.')
```

#### Solution 3 (Stage Abort Rule):
```python
print('When any single task fails 4 consecutive times.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Task Retries**: Default 4 attempts per task.
- **Stage Resilience**: Stage retries if shuffle fetch fails.

### Essential Technical Terms
- `attemptNumber`: Current retry attempt index.
- `Poison Pill`: Deterministic failure record.

### Common Engineering Mistakes
- **Setting maxFailures=1 in production**: Any minor network blip aborts the entire job.

### Interview Questions on This Topic
- **Q: How many times will Spark retry a failed task by default?**
  - *A*: 4 times (spark.task.maxFailures = 4).


### More Interview Questions

- **Q: What is the difference between a task retry and a stage retry?**
  - *A*: A task retry reruns one failed task; a stage retry reruns the stage after failures that invalidate its outputs, such as lost shuffle data.
- **Q: When do retries not help?**
  - *A*: When the failure is deterministic, such as a malformed record, a code bug, or a missing file; every attempt fails the same way.

### What to Study Next
Open 04_checkpointing.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')